### Imports

In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma

C:\Users\PMLS\AppData\Local\Temp\ipykernel_16824\2664897795.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


### Load PDF Files

In [2]:
pdf_files = [
    "../Data/PDF/oecd_ecommerce_consumer_protection.pdf",
    "../Data/PDF/oecd_online_marketplaces.pdf"
]

pdf_documents = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    pdf_documents.extend(loader.load())

print(f"Loaded {len(pdf_documents)} PDF pages")

Loaded 59 PDF pages


### Load Text Files

In [3]:
text_files = [
    "../data/Text/olist_ecommerce_terms.txt",
    "../data/Text/olist_general_terms.txt",
    "../data/Text/olist_privacy_policy.txt",
    "../data/Text/olist_shipping_terms.txt"
]

text_documents = []

for file in text_files:
    loader = TextLoader(file, encoding="utf-8")
    text_documents.extend(loader.load())

print(f"Loaded {len(text_documents)} text documents")

Loaded 4 text documents


### Combine Documents

In [4]:
documents = pdf_documents + text_documents

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 63


### Split Documents

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 341


### Embedding Model

In [6]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

### Vector DB

In [7]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../chroma_db"
)

### Retriever

In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [9]:
query = "What does Olist's privacy policy say about personal data?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Document {i} ---")
    print(doc.page_content[:1000])
    print()

Retrieved 4 documents

--- Document 1 ---
For information on how OLIST uses information and Personal Data, we recommend that You read this entire Privacy Policy and, if applicable, the Privacy Policy specific to each product You contract.
 

5. Find out what information and personal data OLIST processes
As provided for in the various Terms of Use of the OLIST group, in order to be able to register and use the Platforms and services provided by OLIST, the User must initially provide some of the following commercial data or Personal Data, depending on the product and/or service to be contracted:

Information about individuals:
👉 Full name;
👉 Phone number;
👉 Email address;
👉 Home address;
👉 CPF;
👉 Sex;
👉 Date of Birth;
👉 Scanned copy of personal document and data contained in this document, including, if necessary, analysis of the official photograph.

--- Document 2 ---
For information on how OLIST uses information and Personal Data, we recommend that You read this entire Privacy Policy 

### LLM

In [10]:
llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0
)

### Rag

In [11]:
def rag_answer(question: str) -> str:
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

If the answer is not present in the context, say that the information is not available in the provided documents.
"""

    response = llm.invoke(prompt)

    return response.content

In [12]:
answer = rag_answer(
    "What does Olist's privacy policy say about personal data?"
)

print(answer)

Olist's privacy policy states that it respects the privacy of individuals and commits to protecting their personal data. It outlines that the policy informs users about how personal data is collected, used, and handled, with specific details including information such as full name, phone number, email address, home address, CPF, sex, date of birth, and scanned documents. The policy emphasizes that users should read the entire Privacy Policy and relevant product-specific policies for detailed information on data usage.
